# Downsampling approaches
Exploring & implementing the different approaches to downsampling a genome

## MinHashing
Confirmed downsampling approach from Special Course.

Pros:
- Fast & efficient

Cons:
- Hashing algorithm unavailable; backtracking hash values to kmers is not possible (e.g. feature attributions for NNs)

### Constructing singular signatures

In [1]:
import os, sys 
from manipulations import construct_SM_sketches
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

### Phage Minhash Sketch Construction ###
k = 31
n = 1000
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                      k = k, 
                      outdir = f"PhageMinhash_n{n}_k{k}/", 
                      quiet = False,
                      sourmash_parameters=[n, 0])


### Bacteria Minhash Sketch Construction ###
k = 31
n = 1000
construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                      k = k, 
                      outdir = f"BactMinhash_n{n}_k{k}/", 
                      quiet = False,
                      sourmash_parameters=[n, 0])

Created output directory: PhageMinhash_n1000_k31/
------- Constructing MinHashes -------


Constructing minhashes for all records: 100%|██████████| 23/23 [00:02<00:00, 10.89seq/s]


------- Saving Sketches -------
------- Process Completed -------
Created output directory: BactMinhash_n1000_k31/
------- Constructing MinHashes -------


Constructing minhashes for all records: 100%|██████████| 280/280 [16:25<00:00,  3.52s/seq]

------- Saving Sketches -------
------- Process Completed -------


### Constructing multiple signatures

In [ ]:
from manipulations import construct_SM_sketches
import os, sys 
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"

for k in [6, 9, 12, 15, 18, 24]:
    for n in [50, 100, 500, 1000, 5000]:
        #Phages minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/phage_cleaned.fasta", 
                            k = k, 
                            outdir = f"PhageMinhash_n{n}_k{k}_rev/", 
                            quiet = False,
                            sourmash_parameters=[n, 0],
                            include_reverse=True)
        
        #Bacteria minhash sketches
        construct_SM_sketches(fasta = raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta", 
                        k = k, 
                        outdir = f"BactMinhash_n{n}_k{k}_rev/", 
                        quiet = False,
                        sourmash_parameters=[n, 0],
                        include_reverse=True)

## Novel decompisition method
Develop a new method, where i can backtrack the decomposed integers, to its original kmer sequences

Encoder Process:
1) encode the forward k-mer to bit-level integer
2) encode its reverse complement 
3) keep only the smaller of the two

Decomposition Process:
1) divide genome into kmer of size k
2) keep only every x entry, where x = n/genome_kmer_size (n = sig size)
3) save to disk

### Encoder

In [2]:
from decompositions import KmerCodec
codec = KmerCodec()
my_kmer = "GATCGACT"
k_size = len(my_kmer)

# 1. Decompose to integer
encoded_val = codec.encode_with_revcomp(my_kmer)
print(f"Original: {my_kmer}")
print(f"Integer representation: {encoded_val}") # 8864 in decimal

# 2. Backtrack to sequence
decoded_val = codec.decode(encoded_val, k_size)
print(f"Backtracked: {decoded_val}")

Original: GATCGACT
Integer representation: 11661
Backtracked: AGTCGATC


### Decomposition

In [3]:
from decompositions import Decompose
raw_data_path = "../raw_data/"
data_prod_path = "../data_prod/"
k = 12
n = 400

# The 'with' block handles the creation and deletion of tmp automatically
with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="phage", sourmash_like=True) as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/phage_cleaned.fasta"):
        print(line)

with Decompose(k=k, n=n, codec=codec, output_dir=data_prod_path+"encoded_sketches/", entity_type="bact", sourmash_like=True) as decomposer:
    for line in decomposer.decompose(raw_data_path+"phagehost_KU/bacteriaKU_cleaned.fasta"):
        print(line)

Initialized Decompose with k=12, n=400, entity_type='phage', sourmash_like=True


Processing phage FASTA: 23rec [00:00, 153.94rec/s]


Initialized Decompose with k=12, n=400, entity_type='bact', sourmash_like=True


Processing bact FASTA: 280rec [00:50,  5.59rec/s]
